# ELBO Components and Beta Weight

Chapter 14: Information Theory · *Mathematics for AI and Machine Learning* by Fuheng Wu

Explore the β-VAE trade-off: reconstruction vs KL penalty, with waterfall decomposition and ELBO vs β.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-14/elbo-components/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import math

import torch
from torch.distributions import Normal, kl_divergence

torch.manual_seed(0)

# The demo's defaults: reconstruction term, KL term and beta
recon, kl, beta = -2.4, 1.6, 1.0
print(f"ELBO = recon - beta * KL = {recon} - {beta} * {kl} = {recon - beta * kl}")
for name, (r, k, b) in {"vae": (-2.4, 1.6, 1.0), "beta-vae": (-2.4, 1.6, 2.5), "ae-like": (-1.2, 3.2, 0.3)}.items():
    print(f"  preset {name:8s}: ELBO = {r - b * k:+.3f}")

# ── Where the two terms come from, for one data point x ─────────────────────
# Model: z ~ N(0, 1), x | z ~ N(z, 0.7^2). Encoder q(z | x) = N(m, s^2).
x, sx = 1.2, 0.7
prior = Normal(torch.tensor(0.0, dtype=torch.float64), torch.tensor(1.0, dtype=torch.float64))
m = torch.tensor(0.6, dtype=torch.float64, requires_grad=True)
log_s = torch.tensor(math.log(0.8), dtype=torch.float64, requires_grad=True)


def elbo_terms(n=200000):
    q = Normal(m, log_s.exp())
    z = q.rsample((n,))                                   # reparameterized: gradients flow
    rec = Normal(z, sx).log_prob(torch.tensor(x, dtype=torch.float64)).mean()
    return rec, kl_divergence(q, prior)


rec_t, kl_t = elbo_terms()
print(f"\nE_q[log p(x|z)] = {rec_t.item():.4f},  KL(q || p(z)) = {kl_t.item():.4f}")

# ── Maximize the ELBO for several beta: larger beta pulls q toward the prior ─
print("\n beta   q mean   q std    recon     KL")
for b in (0.0, 0.5, 1.0, 2.5, 5.0):
    with torch.no_grad():
        m.fill_(0.0); log_s.fill_(0.0)
    opt = torch.optim.Adam([m, log_s], lr=0.05)
    for _ in range(400):
        opt.zero_grad()
        r, k = elbo_terms(2000)
        (-(r - b * k)).backward()
        opt.step()
    r, k = elbo_terms()
    print(f"{b:4.1f}   {m.item():+.3f}   {log_s.exp().item():.3f}   {r.item():+.4f}   {k.item():.4f}")
# At beta = 1 the optimum is the exact posterior N(x / (1 + sx^2), sx^2 / (1 + sx^2))
print("exact posterior: mean", x / (1 + sx ** 2), " std", math.sqrt(sx ** 2 / (1 + sx ** 2)))

ELBO = recon - beta * KL = -2.4 - 1.0 * 1.6 = -4.0
  preset vae     : ELBO = -4.000
  preset beta-vae: ELBO = -6.400
  preset ae-like : ELBO = -2.160

E_q[log p(x|z)] = -1.5843,  KL(q || p(z)) = 0.2231

 beta   q mean   q std    recon     KL


 0.0   +1.200   0.062   -0.5662   3.0024
 0.5   +0.962   0.444   -0.8228   0.8722


 1.0   +0.803   0.576   -1.0641   0.5395
 2.5   +0.531   0.739   -1.5767   0.2163


 5.0   +0.339   0.852   -2.0585   0.0806
exact posterior: mean 0.8053691275167785  std 0.5734623443633283
